# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking (using Binary Classification)

Explanation in simple words:
We want to figure out if a webpage's traffic is dropping (Yes or No — this is the "Classification" part). But more importantly, the content team can't fix thousands of pages at once. We need the model to score them and sort them from worst to best so the team knows exactly which pages to fix first (this is the "Ranking" part).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. My lane as an ML task (type)

import pandas as pd
import numpy as np

# Load the data slice for this lane
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Build the Yes/No classification label: is traffic trending down?
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Confirm the task type in numbers: how many pages fall into each class?
print("Task Type: Ranking (via a Binary Classification score)")
print()
print("Class balance (is_declining_label):")
print(df["is_declining_label"].value_counts())
print(f"\nShare of pages currently declining: {df['is_declining_label'].mean():.1%}")

# Show that a plain Yes/No split isn't enough to prioritize thousands of pages —
# ranking is needed to turn the binary label into an ordered worklist.
declining_count = df["is_declining_label"].sum()
print(f"\nPages flagged as declining: {declining_count}")
print("A binary Yes/No alone can't tell the content team which of these "
      f"{declining_count} pages to fix first — that's why we rank them.")

Task Type: Ranking (via a Binary Classification score)

Class balance (is_declining_label):
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Share of pages currently declining: 54.2%

Pages flagged as declining: 16262
A binary Yes/No alone can't tell the content team which of these 16262 pages to fix first — that's why we rank them.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Column: is_declining_label

Explanation in simple words:
The exact thing our AI is trying to predict is a simple Yes (1) or No (0): Is this page's traffic trending down?

We want to find pages that are "outdated," but a computer can't easily read a page and know if it feels old. However, a computer can see if a page is losing traffic. So, we use "losing traffic" as our substitute (or "proxy") to flag pages that probably need to be rewritten

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Target or proxy

# Target column: is_declining_label (already built in Section 1, shown again here for clarity)
print("Target Column: is_declining_label")
print()

# What we WANT to predict: "is this page outdated?" — but that's not directly observable.
# What we CAN predict: "is this page's traffic trending down?" — an observed, measurable proxy.
print("What we actually want (label unavailable): page feels outdated")
print("What we use instead (observed proxy):     traffic is trending down")
print()

# Show the proxy rule applied on top of the observed metric
print("Proxy rule: is_declining_label = 1 if trend_direction == 'down', else 0")
print()
print(df["is_declining_label"].value_counts())
print(f"Share flagged by the proxy: {df['is_declining_label'].mean():.1%}")

# Sanity check: look at the raw signal the proxy is built from
df[["trend_direction", "is_declining_label"]].head(10)

Target Column: is_declining_label

What we actually want (label unavailable): page feels outdated
What we use instead (observed proxy):     traffic is trending down

Proxy rule: is_declining_label = 1 if trend_direction == 'down', else 0

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Share flagged by the proxy: 54.2%


,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric: Precision@50

Explanation in simple words:
How do we know if our model did a good job? The content team only has time to update about 50 pages a month. So, if we hand them a list of our "Top 50" worst pages, we want to know: how many of those 50 pages are actually losing traffic?

If 45 out of those 50 pages are dropping, the model is highly accurate and useful. If only 10 are dropping, the model isn't helping. Measuring how many we got right in that top group is our "Precision@50" score

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Success metric

# Success Metric: Precision@50
print("Success Metric: Precision@50")
print()

# Define the metric: out of the top 50 pages we hand to the content team,
# what share are actually declining?
def precision_at_k(df, score_col, label_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

# No real model score exists yet (that comes later) — check the metric
# works using a naive stand-in score: staleness (days since last update).
naive_score_col = "days_since_last_update"
naive_precision = precision_at_k(df, naive_score_col, "is_declining_label", k=50)

print(f"Naive Precision@50 (score = {naive_score_col}): {naive_precision:.1%}")
print(f"Out of the top 50 pages sorted by '{naive_score_col}', "
      f"{naive_precision:.0%} are actually declining.")

# Compare against the base rate — Precision@50 only means something
# if it beats what you'd get by picking pages at random
base_rate = df["is_declining_label"].mean()
print(f"\nBase rate (declining share, all pages): {base_rate:.1%}")
print(f"Lift over base rate: {naive_precision - base_rate:+.1%}")

# Action this metric supports: this Top-50 list IS the deliverable —
# it becomes the monthly content-refresh queue for the writing team.
print("\nAction supported: monthly Top-50 refresh queue handed to content team.")

Success Metric: Precision@50

Naive Precision@50 (score = days_since_last_update): 52.0%
Out of the top 50 pages sorted by 'days_since_last_update', 52% are actually declining.

Base rate (declining share, all pages): 54.2%
Lift over base rate: -2.2%

Action supported: monthly Top-50 refresh queue handed to content team.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
What "Unit of Analysis" means in simple words:
It is just a fancy way of asking, "What does a single row in our spreadsheet represent?" For this project, one row equals one webpage.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# 1. Load the data file from the starter folder
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 2. Create the target column (Yes/No if it is declining)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# 3. Print out what one row represents
print("Unit of Analysis: 1 row = 1 Webpage URL")
print(f"Total pages in dataset: {df.shape[0]}")

# 4. Show the first 5 rows so we can see what the model will learn from
columns_to_show = ["days_since_last_update", "impressions_90d", "ctr", "is_declining_label"]
df[columns_to_show].head()

Unit of Analysis: 1 row = 1 Webpage URL
Total pages in dataset: 30000


,days_since_last_update,impressions_90d,ctr,is_declining_label
0,20,3803,0.76,1
1,25,15320,0.05,1
2,20,12581,0.09,1
3,22,11751,0.49,0
4,14,19140,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule here:

Explanation in simple words:
If we write a rule by hand (like "flag every page that hasn't been updated in exactly 180 days"), it is too rigid. A page updated 179 days ago gets ignored completely, even if its traffic is crashing. Also, a simple rule creates hundreds of ties where many pages get the exact same score, making it impossible to pick a true "Top 50".

An ML model is much smarter. It looks at everything all at once (age, views, and click rates) and learns how they interact. It gives every page a unique percentage score instead of a rigid "yes or no," allowing us to perfectly sort the list from worst to best without running out of signal.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 5. Why ML beats a fixed rule here

# Fixed rule: flag every page not updated in over 180 days
fixed_rule_flag = df["days_since_last_update"] > 180
flagged_by_rule = df[fixed_rule_flag]

print(f"Pages flagged by the 180-day rule: {flagged_by_rule.shape[0]}")
print(f"Of those, share actually declining: {flagged_by_rule['is_declining_label'].mean():.1%}")
print()

# Show the "179 days ago" edge case: pages just under the cutoff, still ignored
near_cutoff = df[(df["days_since_last_update"] >= 150) & (df["days_since_last_update"] <= 180)]
print(f"Pages between 150-180 days (ignored by rule): {near_cutoff.shape[0]}")
print(f"Of those, share actually declining: {near_cutoff['is_declining_label'].mean():.1%}")
print()

# Show the ties problem: many pages sharing the exact same "days_since_last_update" value
print("Most common days_since_last_update values (ties):")
print(df["days_since_last_update"].value_counts().head())
print()

top_tie_value = df["days_since_last_update"].value_counts().idxmax()
tie_count = df["days_since_last_update"].value_counts().max()
print(f"With a fixed rule, all {tie_count} pages at {top_tie_value} days would score identically —")
print("no way to tell which of them to fix first. An ML model breaks these ties")
print("by using impressions, CTR, and other signals together, giving each page")
print("a unique score instead of a rigid Yes/No.")

Pages flagged by the 180-day rule: 174
Of those, share actually declining: 47.1%

Pages between 150-180 days (ignored by rule): 39
Of those, share actually declining: 28.2%

Most common days_since_last_update values (ties):
days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
Name: count, dtype: int64

With a fixed rule, all 11573 pages at 20 days would score identically —
no way to tell which of them to fix first. An ML model breaks these ties
by using impressions, CTR, and other signals together, giving each page
a unique score instead of a rigid Yes/No.


## Self-check

Before you submit, confirm each line honestly:

- [* ] Every section above is filled — markdown thinking AND the code that backs it
- [ *] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ *] No client names, URLs, or private queries anywhere
- [ *] My claims use careful words: observed, measured, directional, decision-support
- [ *] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.